# SDC on a GPU — is there even an incumbent to beat?

The companion notebook (`ssj_gpu_colab.ipynb`) races **symmetric** eigensolvers
against cuSOLVER's `syevd` and loses. That result was clean and the reason was
specific: `syevd` costs only **5.4 gemm-equivalents** at n=2048 on a T4 and
falls toward its flop-ratio floor as n grows, so nothing built out of gemms
fits underneath it.

The **nonsymmetric** problem is a different contest, for two reasons that have
nothing to do with any algorithm:

1. On CPU, `dgeev` costs **83–181 gemm-equivalents** against `dsyevd`'s
   17–25. The incumbent is several times weaker in exactly the unit that
   decided the symmetric race. Spectral divide and conquer has *op-count
   parity* with it — 88 gemm-equivalents against 89, measured at n=400.
2. I expected cuSOLVER to provide **no general nonsymmetric eigensolver at
   all**, which would have left only a host round trip as the baseline. That
   was **stale**: cuSOLVER has gained `xgeev`, and `cupy.linalg.eig` calls it
   directly on the device (`cupy/linalg/_eigenvalue.py` → `_geev` →
   `cusolver.xgeev`, no host transfer). So there IS a genuine device
   incumbent — it is simply a much weaker one than `syevd` was on the
   symmetric side, and it beats a host round trip by 1.8×–2.0× at n ≥ 512.
   Cell 3 checks this at runtime rather than trusting either claim, and still
   reports the round trip because it bounds what you pay by keeping the
   problem on the CPU.

SDC also has a property none of the symmetric methods here do: every
transformation it applies is an **orthogonal similarity**, so it has no basin
condition whatsoever. IPT, SSJ and the shears each stall or diverge on
non-normal input; SDC does not care.

## What this notebook decides

**Cell 1 is the one measurement that has never been run**, and it settles a
question left open on the CPU. SDC's far-field step is one matrix *inverse*;
the alternative maps trade that inverse for gemms. Whether the trade pays is
set entirely by `inv/gemm`, which is a property of the hardware and the
library — not of the algorithm. On this CPU it is **3.90**, which sits between
the two thresholds and is why Halley measured as *noise* rather than as a win
or a loss (OPTIMIZATION_LOG #41). A GPU should push that ratio up, because gemm runs at
peak while `getrf`/`getri` are panel-bound. Cell 1 measures it and prints the
verdict; cell 3 then races the variants instead of trusting the prediction.

**Cell 3 also re-opens the notebook's own headline results.** #41 found that
`dgeev` takes an **88% cache-aliasing penalty at exactly n=512** on the CPU,
which flattered every power-of-two comparison — off the powers of two, the best
CPU variant *lost* 1.35× at n=500 where it appeared to *win* 1.31× at n=512.
This notebook's wins (1.39× at n=256, 1.13× at n=512) are both at powers of
two, and whether `xgeev` has an analogous penalty was never established. Cell
3a measures it directly, and 3b brackets every size with an odd neighbour.

Everything is self-contained; the repository is private, so there is nothing
to clone.


In [ ]:
# =====================================================================
#  1 - The probe: what do SDC's OWN operations cost on this card?
# =====================================================================
# READ THIS CELL'S OUTPUT BEFORE ANYTHING ELSE. It is the deciding measurement
# for the one lever the CPU campaign expects to TRANSFER to a GPU, and it has
# never been run.
#
# The symmetric notebook probes the fp32:fp64 ratio, because mixed precision is
# what that family trades on. SDC trades on something else: its far-field step
# is one matrix INVERSE (the gemm is gated away, see #30 in cell 2) and its
# split needs one QR. So the numbers that predict SDC's cost here are those
# three, in gemm-equivalents.
#
# CPU reference, RE-MEASURED at N=1000 (one gemm = 1). The old numbers quoted
# here were stale by twenty attempts and mis-sized two levers (OPTIMIZATION_LOG #41):
#
#                        carried   measured
#     inverse               5.35       3.90    27% cheaper than assumed
#     QR (full)             7.74       8.45
#     pivoted QR              --      11.04    the worst kernel in the method
#     dgeev                 93.9      82.65
#
# WHY inv/gemm IS THE NUMBER THAT MATTERS. Halley (Pade[1,1]) replaces some
# Newton steps with gemms: it needs a fraction rho of Newton's step count but
# pays 3 extra gemms per step, so it wins exactly when
#
#     inv/gemm  >  3 * rho / (1 - rho)
#
# Measured step ratios on CPU: rho = 0.62 on Ginibre (15->10, 13->8), 0.5-0.55
# on near-symmetric and symmetric (7->5, 8->4, 9->5). That puts the threshold
# at 4.9 on Ginibre and 3.0-3.7 on near-symmetric. CPU measures inv/gemm =
# 3.90, i.e. BELOW the Ginibre threshold and ABOVE the near-symmetric one --
# and that is precisely the split #41 measured end to end: 0.92x-1.01x on
# Ginibre, 1.11x-1.13x on near-symmetric. The model reproduces both verdicts,
# so it is worth trusting on a new substrate.
#
# On a GPU gemm runs at or near peak while getrf/getri are panel-bound and
# latency-exposed, so inv/gemm should be LARGER here -- possibly far larger.
# If it is, Halley stops being noise and the gemm-only endgame orders become
# attractive too. This cell measures the ratio and prints the verdict; cell 3
# then races the variants instead of taking the prediction on faith.
#
# NON-POWERS-OF-TWO ARE IN THE SIZE LIST DELIBERATELY. #41 found dgeev takes
# an 88% cache-aliasing penalty at exactly n=512 and 19% at n=1024, which
# flattered every power-of-two comparison in #36-#40. Whether cuSOLVER and
# cuBLAS do the same on this card is unknown, so the probe brackets each power
# of two with a nearby odd size and prints the gap.

import shutil, subprocess, time
import numpy as np

if shutil.which("nvidia-smi") is None:
    raise SystemExit(
        "No GPU on this runtime (nvidia-smi is not installed).\n"
        "Runtime > Change runtime type > Hardware accelerator: GPU, "
        "then run this cell again.")
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
     "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not smi:
    raise SystemExit("nvidia-smi reports no GPU. Switch the runtime to GPU.")
print("GPU:", smi)

try:
    import cupy as cp
except Exception as e:
    raise SystemExit(f"cupy is unavailable on this runtime: {e}")
print("cupy", cp.__version__, "| numpy", np.__version__)


def _sync():
    cp.cuda.Device().synchronize()


def _best(fn, reps=5):
    fn(); _sync()
    b = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); _sync()
        b = min(b, time.perf_counter() - t0)
    return b


print(f"\n  {'n':>6}{'gemm ms':>10}{'ns/n^3':>9}{'inverse':>10}{'slogdet':>10}"
      f"{'QR':>10}   <- ratios are gemm-equivalents")
OPS = {}
for n in (500, 512, 1000, 1024, 2000, 2048):
    rng = cp.random.default_rng(0)
    A = rng.standard_normal((n, n), dtype=cp.float64)
    A = A + n * cp.eye(n)                     # keep it comfortably invertible
    tg = _best(lambda: A @ A)
    ti = _best(lambda: cp.linalg.inv(A), reps=3)
    td = _best(lambda: cp.linalg.slogdet(A), reps=3)
    tq = _best(lambda: cp.linalg.qr(A), reps=3)
    OPS[n] = dict(gemm=tg, inv=ti / tg, slogdet=td / tg, qr=tq / tg,
                  gemm_n3=tg * 1e9 / n ** 3)
    print(f"  {n:6d}{tg*1e3:10.2f}{OPS[n]['gemm_n3']:9.1f}{ti/tg:10.2f}"
          f"{td/tg:10.2f}{tq/tg:10.2f}")

# --- is there a power-of-two penalty on this card, as there is on the CPU? ---
print("\n=== power-of-two check (gemm ns/n^3; flat means no aliasing penalty)")
worst = 0.0
for lo, hi in ((500, 512), (1000, 1024), (2000, 2048)):
    a, b = OPS[lo]["gemm_n3"], OPS[hi]["gemm_n3"]
    pen = (b / a - 1.0) * 100.0
    worst = max(worst, pen)
    print(f"  n={lo:5d} {a:7.1f}   vs   n={hi:5d} {b:7.1f}"
          f"   -> {pen:+6.1f}% at the power of two")
print(f"  worst gemm penalty {worst:+.1f}%.", end=" ")
print("gemm looks size-insensitive here; the open question is whether\n"
      "  cuSOLVER's xgeev does the same -- cell 3 measures THAT, because it is\n"
      "  what #36/#38's n=256 and n=512 wins actually rest on."
      if worst < 10 else
      "gemm ITSELF is penalised at powers of two on\n"
      "  this card, so every power-of-two timing in #36-#40 needs re-reading.")

# --- the Halley verdict, from the measured ratio rather than from theory ---
inv_ratio = float(np.median([OPS[n]["inv"] for n in (1000, 1024, 2000, 2048)]))
print(f"\n=== the lever this cell decides: inv/gemm = {inv_ratio:.2f} "
      f"(CPU measured 3.90)")
for name, rho in (("Ginibre", 0.62), ("near-symmetric", 0.53)):
    thr = 3.0 * rho / (1.0 - rho)
    verdict = "HALLEY SHOULD WIN" if inv_ratio > thr else "Halley should lose"
    margin = inv_ratio / thr
    print(f"  {name:>16}: rho={rho:.2f} -> threshold {thr:.2f}   "
          f"{verdict}  ({margin:.2f}x the threshold)")
print("""
  If the margin is within ~10% of 1.00 the prediction is inside noise and only
  cell 3's end-to-end race settles it -- that is exactly how Halley came out
  NEUTRAL on CPU rather than clearly refuted. If the margin is 1.5x or more,
  the far-field map is the wrong one for this hardware and cell 3 should show
  it plainly.

  A far-field NEWTON step costs one inverse plus O(n^2) (the convergence gemm
  is gated). A HALLEY step costs one inverse plus 3 gemms. Multiply your
  measured per-step cost by 12-20 steps and compare it to what cell 3 reports
  for cupy.linalg.eig BEFORE believing any timing: if the product already
  exceeds it, SDC cannot win here and the rest is a foregone conclusion.

  ALSO NOTE the inverse-free variant (Bai-Demmel-Gu) is NOT offered. It was
  refuted analytically and unconditionally on CPU: one IRS step is 13.33n^3
  against Newton's 2n^3, so 15 steps is 200n^3, which at PERFECT gemm
  efficiency already exceeds dgeev's whole 93.9 gemm-equivalents. That
  refutation is in flops, so it survives the change of substrate -- but its
  stability claim also checked out (it converges on cond(V)=1e6 where the
  shipped iteration stalls), so it is a fallback, not a speed lever.""")


In [ ]:
# =====================================================================
#  2 - SDC, backend-agnostic (same code path on numpy and cupy arrays)
# =====================================================================
# Ported from ssj.sdc (OPTIMIZATION_LOG #24-31, #39, #41) and validated on the NumPy
# path against that implementation across Ginibre, planted-real, near-symmetric
# and companion matrices.
#
# Findings from the CPU campaign built in, each of which cost a tick to learn:
#
#   #31  THE NEWTON->NEWTON-SCHULZ HANDOFF MUST BE TESTED IN THE RIGHT NORM.
#        NS converges only inside ||I - X^2||_2 < 1. The iteration's natural
#        progress measure is ||I - X^2||_F/sqrt(n), an RMS quantity that sits
#        far BELOW the operator norm, so a fixed threshold on it tests the
#        wrong thing -- and how wrong depends on the spectrum. At dev < 0.9
#        (swept on Ginibre alone) a SYMMETRIC matrix entered NS outside its
#        region and never converged: 2712 ms against dgeev's 48 ms at n=400,
#        13448 ms at n=800. Since ||M||_2 <= ||M||_F, gating on
#        ||I - X^2||_F < 1 is guaranteed safe -- a 1/sqrt(n) SCALING LAW.
#
#   #30  THE FAR-FIELD CONVERGENCE GEMM IS OPTIONAL. A Newton step is X^2
#        (2n^3) plus the inverse, and half of it is a gemm whose only job is
#        the convergence test. But Delta = (X^-1 - X)/2 = X^-1(I - X^2)/2, so
#        the update norm is the same signal for O(n^2). Measured: Delta tracks
#        dev/2 to two digits through the endgame and is never small while dev
#        is large, so gating on it reproduces the same handoff while forming
#        X^2 once or twice instead of 8-11 times.
#
#   #28  THE LEAF IS 3n/5, NOT n/2. The centred split returns r = trace(P),
#        which lands near n/2 but essentially never ON it, so a leaf of n/2
#        sends one half back a few rows too big and buys a whole second
#        full-size sign iteration. Worth 1.10x-1.16x.
#
#   #39  THE SPLIT GATE IS 1e-11, NOT 1e-6. See _split_once.
#
#   #41  THE FAR-FIELD MAP IS A HARDWARE CHOICE, NOT AN ALGORITHMIC ONE, which
#        is why far= and ns_order= are parameters here rather than constants.
#        See sign_iterate.

import time
import numpy as np


def _am(A):
    if type(A).__module__.partition(".")[0] == "cupy":
        import cupy
        return cupy
    return np


_G_CACHE = {}


def _inv_and_logdet(X, scaling):
    """Inverse, and log|det| only if the scaling actually needs it.

    THIS IS A MEASURED CHOICE, not an obvious one. Determinantal scaling
    mu = |det X|^(-1/n) needs log|det|, which costs a SECOND factorization
    unless it is shared -- and sharing it via lu_solve against a full identity
    (2n^3) is no cheaper in flops than getrf+getri (2n^3) plus a getrf for the
    determinant (2n^3/3). So on GPU the honest options are 2.67n^3 with
    scaling or 2n^3 without.

    Is the scaling worth 33%? Measured on CPU, iteration counts with
    determinantal scaling against unscaled Newton:

        Ginibre  n=200/400/800:  16/14/17  vs  17/15/19
        near-sym n=200/400/800:  23/22/25  vs  23/22/25

    So unscaled costs 0-13% more ITERATIONS to save 33% per far-field step.

    RACED ON A GPU (OPTIMIZATION_LOG #36) AND UNSCALED WINS EVERYWHERE, so it is now
    the default: 2.14x/1.11x/1.26x faster at n=256/512/1024 with leaf=host,
    and 1.25x/1.51x/1.55x with leaf=deep, at unchanged accuracy. The n=256
    gain far exceeds the 33% the flop count predicts, which points at the
    other cost of the scaled branch: reading log|det| back to the host is a
    SYNC per Newton step, and at small n that dominates the arithmetic.
    """
    xp = _am(X)
    n = X.shape[0]
    if scaling == "none":
        return xp.linalg.inv(X), 0.0
    sgn, lad = xp.linalg.slogdet(X)
    return xp.linalg.inv(X), float(lad)


# Newton-Schulz family: X <- X * p(X^2), where p truncates (1-t)^{-1/2} at
# t = I - X^2. All are GEMM-ONLY and all converge only inside ||I-X^2||_2 < 1,
# so ns_order changes the ENDGAME cost, never the handoff test.
#
#   order 3: X(3I - X^2)/2                        2 gemms, quadratic in E
#   order 5: X(15I - 10X^2 + 3X^4)/8              3 gemms, cubic in E
#   order 7: X(35I - 35X^2 + 21X^4 - 5X^6)/16     4 gemms, quartic in E
#
# Work per digit is log(p-1)/gemms: 0.347, 0.366, 0.347 -- order 5 is best by
# about 5%, which is small because the endgame is only 2-5 steps. Measured
# endgame step counts on CPU (order 3 / 5 / 7):
#
#     Ginibre  n=200  2/2/1     near-sym n=200  4/2/2     symmetric n=200 4/3/2
#     Ginibre  n=400  2/2/1     near-sym n=400  5/3/3     symmetric n=400 4/3/2
#
# So the order buys real steps but on a short phase. It is exposed because on
# a GPU gemms are the cheap primitive and this is the only part of the method
# that is pure gemm.
_NS_COEF = {3: (1.5, -0.5),
            5: (15 / 8, -10 / 8, 3 / 8),
            7: (35 / 16, -35 / 16, 21 / 16, -5 / 16)}


def _ns_step(X, X2, eye, order):
    """One gemm-only Newton-Schulz step of the given order. X2 = X @ X is
    already formed by the convergence test, so it is never recomputed."""
    c = _NS_COEF[order]
    P = c[0] * eye + c[1] * X2
    if order >= 5:
        X4 = X2 @ X2
        P = P + c[2] * X4
        if order >= 7:
            P = P + c[3] * (X4 @ X2)
    return X @ P


def sign_iterate(X, ns_frob=1.0, tol=1e-12, max_iter=60, scaling="none",
                 far="newton", ns_order=3):
    """Matrix sign: a far-field map while far away, Newton-Schulz once close.

        Newton (Pade[0,1])  X <- (mu X + mu^-1 X^-1)/2      1 inv, quadratic
        Halley (Pade[1,1])  X <- X(3I + X^2)(I + 3X^2)^-1   1 inv + 3 gemms,
                                                            cubic
        Newton-Schulz       X <- X p(X^2)                   gemm-only, see
                                                            _NS_COEF

    Returns (S, iters, ok, n_far, n_ns).

    WHICH FAR-FIELD MAP IS A HARDWARE QUESTION (OPTIMIZATION_LOG #41 + addendum).
    Halley needs a fraction rho of Newton's step count and pays 3 extra gemms
    per step, so it wins exactly when inv/gemm > 3*rho/(1-rho). Measured step
    counts, Newton -> Halley:

        Ginibre    n=200  15 -> 10      n=400  13 -> 8     rho ~ 0.62
        near-sym   n=200   7 ->  5      n=400  13 -> 8     rho ~ 0.55
        symmetric  n=200   8 ->  4      n=400   9 -> 5     rho ~ 0.53

    giving a threshold of 4.9 on Ginibre and 3.0-3.7 on near-symmetric. This
    CPU measures inv/gemm = 3.90 -- between the two -- and that is exactly the
    split #41 measured end to end: 0.920x-1.010x on Ginibre, 1.11x-1.13x on
    near-symmetric. The model predicts both signs correctly, which is why it is
    worth extrapolating.

    A GPU should push inv/gemm UP (gemm at peak, getrf/getri panel-bound and
    latency-exposed), so Halley is expected to win here. Cell 1 prints the
    measured ratio and the verdict; cell 3 races it. Accuracy is unchanged
    either way -- validated on the NumPy path at n=200 and 400 across Ginibre,
    near-symmetric and symmetric input, ||A21||/||A|| landing 5.1e-15..2.0e-12
    for Halley against 7.6e-15..5.4e-13 for Newton, with identical split ranks.

    NOTE Halley ignores `scaling`: the Pade[1,1] map is already scale-invariant
    in the sense that matters here, and it has no det-based accelerator to
    apply. That also means it never syncs log|det| to the host, which is a
    second, separate reason to expect it to do relatively better on a GPU.
    """
    xp = _am(X)
    n = X.shape[0]
    sqn = np.sqrt(n)
    thresh = ns_frob / sqn                       # #31: a scaling law
    eye = xp.eye(n, dtype=X.dtype)

    nrm = float(xp.linalg.norm(X, ord="fro")) / sqn
    if nrm == 0.0:
        return X, 0, False, 0, 0
    X = X / nrm

    delta_prev = np.inf
    since_check = 0
    n_far = n_ns = 0
    for it in range(1, max_iter + 1):
        if delta_prev < thresh or since_check >= 8:   # #30: gate the gemm
            since_check = 0
            X2 = X @ X
            dev = float(xp.linalg.norm(X2 - eye, ord="fro")) / sqn
            if not np.isfinite(dev):
                return X, it, False, n_far, n_ns
            if dev < tol:
                return X, it, True, n_far, n_ns
            if dev < thresh:
                X = _ns_step(X, X2, eye, ns_order)
                n_ns += 1
                delta_prev = 0.0
                continue
        else:
            since_check += 1

        if far == "halley":
            # X <- X(3I + X^2)(I + 3X^2)^-1.  One inverse, 3 gemms, cubic.
            X2 = X @ X
            num = X @ (3.0 * eye + X2)
            den = eye + 3.0 * X2
            try:
                Xn = num @ xp.linalg.inv(den)
            except Exception:
                return X, it, False, n_far, n_ns
        else:
            Xi, lad = _inv_and_logdet(X, scaling)
            if scaling != "none" and not np.isfinite(lad):
                return X, it, False, n_far, n_ns     # singular iterate
            mu = 1.0 if scaling == "none" else np.exp(-lad / n)
            Xn = 0.5 * (mu * X + Xi / mu)

        delta_prev = float(xp.linalg.norm(Xn - X, ord="fro")) / sqn
        X = Xn
        n_far += 1
        if not np.isfinite(delta_prev):
            return X, it, False, n_far, n_ns
    return X, max_iter, False, n_far, n_ns


def _split_once(A, shift, ns_frob=1.0, tol=1e-12, scaling="none",
                gate=1e-11, far="newton", ns_order=3, st=None):
    """One spectral split at Re(z) = shift. Returns (B, r) or (None, code)."""
    xp = _am(A)
    n = A.shape[0]
    S, its, ok, nf_, ns_ = sign_iterate(A - shift * xp.eye(n, dtype=A.dtype),
                                        ns_frob=ns_frob, tol=tol,
                                        scaling=scaling, far=far,
                                        ns_order=ns_order)
    if st is not None:
        st["newton"] += nf_; st["ns"] += ns_; st["sign_calls"] += 1
    if not ok:
        return None, -2
    P = 0.5 * (xp.eye(n, dtype=A.dtype) + S)
    r = int(np.rint(float(xp.trace(P))))
    if r <= 0 or r >= n:
        return None, -1

    # Randomized range-finder instead of pivoted QR (cupy's qr has no
    # pivoting). The FIRST r columns must come from P and the rest from I - P,
    # IN THAT ORDER: OPTIMIZATION_LOG #24 records building the basis from a pivoted QR
    # of [P, I-P] instead, whose column reordering destroys the range
    # separation and reported a bogus ||A21|| = 2.6e-01 on a symmetric matrix.
    #
    # This was a forced port change that turned out to be an IMPROVEMENT: #41
    # re-measured the kernels and pivoted QR runs at 6.0% of gemm rate, 11.04
    # gemm-equivalents, making it the worst-performing primitive in the whole
    # method -- worse per flop than dgeev. The range-finder is 1 gemm + one
    # unpivoted dgeqrf = 9.45. The CPU version still uses dgeqp3; the saving
    # is only 1.3% of the total there, which is why it has not been changed.
    key = (n, xp.__name__)
    if key not in _G_CACHE:
        _G_CACHE[key] = xp.asarray(
            np.random.default_rng(0x5D1).standard_normal((n, n)))
    G = _G_CACHE[key].astype(A.dtype, copy=False)
    Y = xp.empty((n, n), dtype=A.dtype)
    Y[:, :r] = P @ G[:, :r]
    Y[:, r:] = G[:, r:] - P @ G[:, r:]
    Q = xp.linalg.qr(Y)[0]
    B = Q.T @ (A @ Q)

    # The (2,1) block is zero in exact arithmetic; how far it misses IS the
    # split's backward error. For nonsymmetric A it scales with the OBLIQUE
    # projector norm ||P||, which equals 1 only in the symmetric case
    # (#24: ||P|| 3.2 -> 1.5e5 as cond(X) runs 10 -> 1e6).
    #
    # THE GATE WAS 1e-6 AND THAT WAS FIVE ORDERS TOO LOOSE (OPTIMIZATION_LOG #39). It
    # never fired: measured headroom against what the split actually achieves
    # ran 25892x at n=1024 and 17 MILLION x at n=256, so a genuinely bad split
    # sailed through. Cause, established by instrumenting: at n=1024 an
    # eigenvalue sat 1.83e-04 from the splitting line, the sign function is
    # ill-conditioned there, and ||A21|| degraded to 3.86e-11 -- while other
    # shifts on the SAME matrix reached 1.6e-13..1.1e-12.
    #
    # The retry loop was already the right machinery; only the threshold was
    # wrong. At 1e-11 the n=1024 split is rejected, two retries find a good
    # shift, and the error improves 99x (1.04e-10 -> 1.05e-12). It costs 2.5x
    # the sign work AT THAT SIZE ONLY -- n<=512 never rejects and pays nothing.
    if float(xp.linalg.norm(B[r:, :r], ord="fro")) > \
            gate * float(xp.linalg.norm(A, ord="fro")):
        return None, -3
    return B, r


def _leaf_closed_form(M):
    """1x1 or 2x2 in closed form, on the device. A conjugate pair shares a
    real part, so a vertical cut never separates one -- 2x2 always suffices."""
    xp = _am(M)
    if M.shape[0] == 1:
        return xp.asarray(M[0, 0], dtype=np.complex128).reshape(1)
    a, b, c, d = M[0, 0], M[0, 1], M[1, 0], M[1, 1]
    tr, det = a + d, a * d - b * c
    root = xp.sqrt(xp.asarray(tr * tr / 4.0 - det, dtype=np.complex128))
    return xp.stack([tr / 2.0 + root, tr / 2.0 - root])


def _to_host(A):
    return A if _am(A) is np else A.get()


# Leaf size at or above which the DEVICE solver is used by leaf_solver="auto".
# Measured directly (OPTIMIZATION_LOG #38), and it is a real crossover, not a taste:
#
#     leaf size    host round trip    cupy.linalg.eig    winner
#           128            15.1 ms            54.1 ms    host
#           256            70.1 ms            94.2 ms    host
#           384           154.8 ms           166.6 ms    host
#           512           433.3 ms           246.2 ms    device
#           768           830.6 ms           455.7 ms    device
#
# So host wins THROUGH 384 and device takes over by 512; 450 sits between the
# measured bracket. An earlier value of 384 was too low -- host still won
# there. The shipped leaf of 3n/5 puts leaves at ~n/2, so for n in 512..1024
# this constant is what decides which solver the leaves actually use.
#
# Note how weakly cupy.linalg.eig scales: 128 -> 768 is 6x in size but only
# 8.4x in time, nothing like the 216x a cubic would give. It is dominated by
# fixed cost at these sizes, which is why it LOSES to a host round trip below
# ~450 and wins comfortably above it. Cell 3 re-measures the crossover rather
# than trusting this number on a different card, and now brackets the powers
# of two while doing so (#41: the CPU's dgeev has an 88% penalty at n=512, and
# whether xgeev has one is exactly what these leaf timings would reveal).
LEAF_DEVICE_MIN = 450


def _leaf_solve(A, leaf_solver):
    """Eigenvalues of a leaf block. Three strategies, and which wins depends
    on the block SIZE, not on taste (see LEAF_DEVICE_MIN)."""
    xp = _am(A)
    n = A.shape[0]
    if n <= 2:
        return _leaf_closed_form(A)
    if xp is np:
        return np.asarray(np.linalg.eigvals(A), dtype=np.complex128)
    want_device = (leaf_solver == "device" or
                   (leaf_solver == "auto" and n >= LEAF_DEVICE_MIN))
    if want_device:
        try:
            return xp.asarray(xp.linalg.eig(A)[0], dtype=np.complex128)
        except Exception:
            pass          # no device eig on this CuPy: fall through to host
    return xp.asarray(np.linalg.eigvals(_to_host(A)), dtype=np.complex128)


def _now(xp):
    """Wall clock with a device sync -- ONLY called when profiling, because
    a sync per phase would perturb the very thing the race measures."""
    if xp is not np:
        xp.cuda.Device().synchronize()
    return time.perf_counter()


def sdc_eigvals(A, min_block=None, leaf_solver="auto", ns_frob=1.0,
                tol=1e-12, scaling="none", gate=1e-11, far="newton",
                ns_order=3, profile=False, _depth=0, _st=None):
    """Eigenvalues of a general real matrix by spectral divide and conquer.

    leaf_solver : how leaf blocks are solved.
        "device" -- cupy.linalg.eig on the device, no transfer. SDC then acts
                    as a PRECONDITIONER for the vendor solver: split once,
                    then two half-size device solves.
        "host"   -- copy each leaf to the CPU for numpy.linalg.eigvals.
        "auto"   -- device at or above LEAF_DEVICE_MIN, host below. Neither
                    wins everywhere: at a leaf of 256 the host was 70.4 ms
                    against the device's 86.9 ms, and at 512 the host was
                    458.6 ms against 257.0 ms (OPTIMIZATION_LOG #36 run).
        "deep"   -- recurse to 2x2 and never leave the device. Measured
                    4.4x-16.4x SLOWER on GPU; kept as a control only.
    gate : reject a split whose ||A21||/||A|| exceeds this and retry a
        different shift. 1e-11 is measured (OPTIMIZATION_LOG #39); the historical 1e-6
        never fired and let a 99x-worse split through at n=1024. Raising it
        back to 1e-6 reproduces the old behaviour if you need to compare.
    far : far-field map, "newton" or "halley". A HARDWARE choice -- see
        sign_iterate. Halley needs ~0.53-0.62 of Newton's steps and pays 3
        extra gemms per step; cell 1 prints the break-even for this card.
    ns_order : 3, 5 or 7. Order of the gemm-only Newton-Schulz endgame. Higher
        orders cut endgame steps (4 -> 3 -> 2 typical) at one extra gemm each.
    """
    xp = _am(A)
    n = A.shape[0]
    if _st is None:
        _st = {"splits": 0, "sign_calls": 0, "newton": 0, "ns": 0,
               "leaves": 0, "fallbacks": 0,
               "t_split": 0.0, "t_leaf": 0.0}
    if min_block is None:
        min_block = 2 if leaf_solver == "deep" else max(2, 3 * n // 5)  # #28

    if n <= min_block or _depth >= 64:
        _st["leaves"] += 1
        t0 = _now(xp) if profile else 0.0
        out = _leaf_closed_form(A) if leaf_solver == "deep" and n <= 2 \
            else _leaf_solve(A, leaf_solver)
        if profile:
            _st["t_leaf"] += _now(xp) - t0
        return out, _st

    centre = float(xp.trace(A)) / n
    spread = float(xp.linalg.norm(A, ord="fro")) / np.sqrt(n)
    rng = np.random.default_rng(0xC0FFEE + _depth)
    B, r = None, -1
    t0 = _now(xp) if profile else 0.0
    for attempt in range(12):
        shift = centre if attempt == 0 else centre + spread * float(
            rng.standard_normal()) * 0.5 ** (attempt // 4)
        B, r = _split_once(A, shift, ns_frob, tol, scaling, gate,
                           far, ns_order, _st)
        if B is not None:
            break
    if profile:
        _st["t_split"] += _now(xp) - t0
    if B is None:
        _st["fallbacks"] += 1
        return xp.asarray(np.linalg.eigvals(_to_host(A)),
                          dtype=np.complex128), _st

    _st["splits"] += 1
    kw = dict(min_block=min_block, leaf_solver=leaf_solver, ns_frob=ns_frob,
              tol=tol, scaling=scaling, gate=gate, far=far,
              ns_order=ns_order, profile=profile, _depth=_depth + 1, _st=_st)
    w1, _ = sdc_eigvals(B[:r, :r], **kw)
    w2, _ = sdc_eigvals(B[r:, r:], **kw)
    return xp.concatenate([w1, w2]), _st


print("SDC loaded.  far in {newton, halley} | ns_order in {3, 5, 7}")


In [ ]:
# =====================================================================
#  3 - Is there an incumbent? Then the race.
# =====================================================================
# Accuracy is asserted against dgeev before any time is believed: a routine
# that fails fast looks fast.
#
# FOUR THINGS CHANGED HERE AFTER OPTIMIZATION_LOG #41, and each is a correction to how
# #36-#40 were measured rather than a new idea:
#
# 1. SIZES COME IN PAIRS, 250/256, 500/512, 1000/1024. On CPU, dgeev takes an
#    88% cache-aliasing penalty at exactly n=512 and 19% at n=1024. Off the
#    powers of two the best CPU SDC variant LOST 1.35x at n=500 and 1.54x at
#    n=1000, where at n=512 it appeared to WIN 1.31x -- entirely dgeev's
#    penalty. #36/#38's GPU wins are at n=256 and n=512, both powers of two,
#    so they are exactly the results this bracketing is here to re-read.
#
# 2. THERE IS AN EXPLICIT xgeev POWER-OF-TWO PROBE below. Whether cuSOLVER has
#    an analogous penalty was never established and was never claimed; it is a
#    two-line measurement and it decides whether the campaign's only win over
#    a vendor eigensolver is real.
#
# 3. far=halley IS RACED, not assumed. Cell 1 predicts the winner from
#    inv/gemm; this cell checks the prediction end to end. A per-kernel
#    measurement that is right in isolation and wrong in place is exactly how
#    #41's linsolve lever failed -- correct at 4.47 gemm-equivalents against
#    5.90, then a 1.4x-2.4x LOSS in situ.
#
# 4. NEAR-SYMMETRIC INPUT IS IN THE MATRIX SET. Halley's CPU win was entirely
#    there (1.11x-1.13x) and absent on Ginibre, because the step-count ratio
#    rho differs by class. Racing Ginibre alone would have reproduced #29's
#    original mistake: sweeping a constant on one matrix class and shipping it.

import time


def sync():
    cp.cuda.Device().synchronize()


def timed(fn, reps=3):
    fn(); sync()
    b = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync()
        b = min(b, time.perf_counter() - t0)
    return b


def matched_err(w, v, nrm):
    """Nearest-match, NEVER a lexicographic sort. Sorting by (re, im) ties on
    exact conjugate pairs and reported a spurious dlam of 1.0 earlier in this
    campaign where the true error was 8.0e-15."""
    w = np.asarray(w.get() if hasattr(w, "get") else w, dtype=complex)
    v = list(np.asarray(v, dtype=complex))
    tot = 0.0
    for x in w:
        d = np.abs(x - np.array(v)); i = int(np.argmin(d))
        tot = max(tot, float(d[i])); v.pop(i)
    return tot / nrm


def ginibre(n, seed=2):
    return np.random.default_rng(seed).standard_normal((n, n)) / np.sqrt(n)


def near_sym(n, seed=2, skew=0.05):
    W = np.random.default_rng(seed).standard_normal((n, n)) / np.sqrt(n)
    return (W + W.T) / 2 + skew * (W - W.T) / 2


print("=== does this GPU have a general nonsymmetric eigensolver at all?")
HAVE_GPU_EIG = False
try:
    cp.linalg.eig(cp.asarray(ginibre(64)))
    HAVE_GPU_EIG = True
    print("  cupy.linalg.eig EXISTS -- it becomes the incumbent below")
    print("  (cupy/linalg/_eigenvalue.py -> _geev -> cusolver.xgeev, on the")
    print("   device, no host transfer)")
except Exception as e:
    print(f"  no general eig on the device: {type(e).__name__}: {e}")
    print("  -> the baseline is a HOST ROUND TRIP, transfers included.")
    print("     You cannot get eigenvalues of a device matrix without either")
    print("     a device solver or a copy, so excluding the copy would be")
    print("     measuring a solver that does not exist.")


def host_roundtrip(A):
    return cp.asarray(np.linalg.eigvals(A.get()))


# --------------------------------------------------------------------------
#  3a - does cuSOLVER's xgeev have the CPU's power-of-two penalty?
# --------------------------------------------------------------------------
# This is the cheapest measurement in the notebook and it recolours #36/#38.
if HAVE_GPU_EIG:
    print("\n=== xgeev power-of-two probe (ns/n^3; a spike AT the power of two")
    print("    would mean #36/#38's n=256 and n=512 wins were partly artefact)")
    print(f"  {'n':>6}{'xgeev ms':>11}{'ns/n^3':>10}{'vs neighbour':>14}")
    prev = None
    for n in (250, 256, 264, 500, 512, 520, 1000, 1024):
        An = cp.asarray(ginibre(n, seed=7))
        t = timed(lambda: cp.linalg.eig(An)[0], reps=3)
        norm = t * 1e9 / n ** 3
        rel = "" if prev is None else f"{norm/prev:13.2f}x"
        print(f"  {n:6d}{t*1e3:11.1f}{norm:10.1f}{rel:>14}")
        prev = norm
    print("  -> read the 256 and 512 rows against their 250/264 and 500/520")
    print("     neighbours. Flat means the GPU wins were real.")

# --------------------------------------------------------------------------
#  3b - the race, bracketed around every power of two, on two matrix classes
# --------------------------------------------------------------------------
PAIRS = [(250, 256), (500, 512), (1000, 1024)]
SIZES = [n for pair in PAIRS for n in pair]
CLASSES = (("ginibre", ginibre), ("near-sym", near_sym))

print(f"\n  {'class':>10}{'n':>6}{'method':>24}{'ms':>9}{'vs incumb':>11}"
      f"{'far/ns':>9}{'splits':>7}{'dlam':>10}")
rows = []
for cname, mk in CLASSES:
    for n in SIZES:
        A_h = mk(n)
        A = cp.asarray(A_h)
        nrm = float(np.linalg.norm(A_h, 2))
        wref = np.linalg.eigvals(A_h)

        t_gemm = timed(lambda: A @ A, reps=5)
        t_host = timed(lambda: host_roundtrip(A), reps=3)
        t_inc = t_host
        if HAVE_GPU_EIG:
            t_inc = min(t_inc, timed(lambda: cp.linalg.eig(A)[0], reps=2))
        print(f"\n  {cname} n={n}: fp64 gemm {t_gemm*1e3:.2f} ms | host trip "
              f"{t_host*1e3:.1f} | incumbent {t_inc*1e3:.1f} ms "
              f"= {t_inc/t_gemm:.0f} gemm-eq")

        cands = [("host dgeev round trip", lambda: host_roundtrip(A))]
        if HAVE_GPU_EIG:
            cands.append(("cupy.linalg.eig", lambda: cp.linalg.eig(A)[0]))
        # leaf=auto only in the main race; the leaf sweep is 3c and deep is a
        # control measured 4.4x-16.4x slower on GPU (#36), raced once in 3c.
        for far in ("newton", "halley"):
            cands.append((f"SDC far={far}", lambda f=far: sdc_eigvals(
                A, leaf_solver="auto", far=f)))

        for label, fn in cands:
            try:
                out = fn()
                w, st = out if isinstance(out, tuple) else (out, {})
                sync()
                e = matched_err(w, wref, nrm)
                if not (e < 1e-8):
                    print(f"  {cname:>10}{n:6d}{label:>24}{'':>9}{'':>11}"
                          f"{'':>9}{st.get('splits', 0):7}{e:10.1e}"
                          f"  NOT ACCURATE")
                    continue
                t = timed(fn, reps=2)
                is_sdc = label.startswith("SDC")
                mark = "   <-- BEATS the incumbent" if (
                    is_sdc and t < t_inc) else ""
                fns = (f"{st.get('newton', 0)}/{st.get('ns', 0)}"
                       if st else "--")
                print(f"  {cname:>10}{n:6d}{label:>24}{t*1e3:9.1f}"
                      f"{t_inc/t:10.2f}x{fns:>9}{st.get('splits', 0):7}"
                      f"{e:10.1e}{mark}")
                rows.append((cname, n, label, t, t_inc, t_inc / t, e))
            except Exception as ex:
                print(f"  {cname:>10}{n:6d}{label:>24}  FAILED: "
                      f"{type(ex).__name__}: {ex}")

# --- the power-of-two verdict, computed rather than eyeballed ---
print("\n=== is any SDC win an artefact of the power of two?")
print(f"  {'class':>10}{'method':>18}{'odd n':>8}{'ratio':>8}"
      f"{'pow2 n':>8}{'ratio':>8}{'inflation':>11}")
for cname, _ in CLASSES:
    for label in ("SDC far=newton", "SDC far=halley"):
        for lo, hi in PAIRS:
            g = {r[1]: r[5] for r in rows
                 if r[0] == cname and r[2] == label}
            if lo in g and hi in g:
                print(f"  {cname:>10}{label:>18}{lo:8d}{g[lo]:8.2f}"
                      f"{hi:8d}{g[hi]:8.2f}{g[hi]/g[lo]:10.2f}x")
print("  inflation > 1.15x means the power-of-two number FLATTERS SDC and the")
print("  odd-n column is the honest one. That is what happened on CPU (#41).")

# --------------------------------------------------------------------------
#  3c - leaf policy: where does the crossover actually sit on this card?
# --------------------------------------------------------------------------
print("\n=== leaf solver crossover: host round trip vs device eig, by block")
print(f"  {'m':>6}{'host ms':>12}{'device ms':>12}{'winner':>10}")
for m in (128, 250, 256, 384, 500, 512, 768):
    Am = cp.asarray(ginibre(m, seed=3))
    th = timed(lambda: cp.asarray(np.linalg.eigvals(Am.get())), reps=3)
    try:
        td = timed(lambda: cp.linalg.eig(Am)[0], reps=3)
        print(f"  {m:6d}{th*1e3:12.1f}{td*1e3:12.1f}"
              f"{('host' if th < td else 'device'):>10}")
    except Exception as e:
        print(f"  {m:6d}{th*1e3:12.1f}{'--':>12}{'host':>10} "
              f"({type(e).__name__})")
print("  -> set LEAF_DEVICE_MIN in cell 2 to where device takes over")

# --- full variant sweep, at ONE odd size, to keep the run short ---
NV = 500
print(f"\n=== variant sweep at n={NV} (odd on purpose), near-symmetric --")
print("    the class where Halley won on CPU")
A_h = near_sym(NV)
A = cp.asarray(A_h)
nrm = float(np.linalg.norm(A_h, 2))
wref = np.linalg.eigvals(A_h)
print(f"  {'leaf':>8}{'far':>9}{'ns':>4}{'scale':>7}{'ms':>9}"
      f"{'far/ns it':>11}{'splits':>7}{'dlam':>10}")
for leaf in ("auto", "host", "device", "deep"):
    for far in ("newton", "halley"):
        for order in (3, 5):
            for sc in ("none",):
                fn = (lambda l=leaf, f=far, o=order, s=sc: sdc_eigvals(
                    A, leaf_solver=l, far=f, ns_order=o, scaling=s))
                try:
                    w, st = fn()
                    sync()
                    e = matched_err(w, wref, nrm)
                    if not (e < 1e-8):
                        print(f"  {leaf:>8}{far:>9}{order:4d}{sc:>7}{'':>9}"
                              f"{'':>11}{st['splits']:7}{e:10.1e}"
                              f"  NOT ACCURATE")
                        continue
                    t = timed(fn, reps=2)
                    print(f"  {leaf:>8}{far:>9}{order:4d}{sc:>7}{t*1e3:9.1f}"
                          f"{st['newton']:6d}/{st['ns']:<4d}{st['splits']:7}"
                          f"{e:10.1e}")
                except Exception as ex:
                    print(f"  {leaf:>8}{far:>9}{order:4d}{sc:>7}  FAILED: "
                          f"{type(ex).__name__}")

print("""
  Reading these tables

  THE ROW THAT MATTERS is cupy.linalg.eig, not the host baseline -- where it
  exists it is the real incumbent, and on the run behind OPTIMIZATION_LOG #36 it beat
  the host round trip by 1.8x-2.0x at n>=512. The `vs incumb` column is
  already against the better of the two.

  READ 3a BEFORE 3b. If xgeev spikes at 256 and 512 the way dgeev does at 512
  on the CPU, then #36/#38's 1.39x and 1.13x wins are partly that spike, and
  the odd-n rows of 3b are the only honest ones. If 3a is flat, the wins stand
  and the power-of-two worry is closed.

  THE gemm-eq FIGURE IS MIXED-DEVICE, so do not read it as the campaign's
  unit. The host baseline's numerator is CPU dgeev while the denominator is GPU
  gemm, and leaf=host puts a CPU solve inside SDC's own numerator too. The
  campaign's "SDC needs ~88" was measured CPU-over-CPU. Use the millisecond
  and ratio columns.

  HALLEY IS A HARDWARE BET WITH A STATED THRESHOLD. Cell 1 prints inv/gemm and
  the break-even 3*rho/(1-rho); this cell says whether the bet paid. If Halley
  wins here and lost on CPU, the cause is named in advance rather than fitted
  afterwards -- and if it loses here too, the substrate argument is dead and
  the far-field map is simply Newton.

  leaf=device makes SDC a PRECONDITIONER for the vendor solver rather than a
  replacement: one split on the device, then two half-size device eigs. That
  wins whenever the split costs less than the difference between one full solve
  and two half solves -- not a given, because cupy.linalg.eig measured only
  3.07x between n=512 and n=1024, not the 8x a cubic would give, so halving n
  saves far less than it looks.

  leaf=deep is a CONTROL, not a candidate. #25 measured deep recursion 4x
  slower on CPU and the expectation here was that a device without geev would
  flip that. It did not -- #36 measured deep 4.4x-16.4x SLOWER on GPU, because
  the premise was wrong twice over: cupy.linalg.eig exists, and the bottom
  levels issue hundreds of tiny kernels (526 splits at n=1024).

  scale=none is the default and won everywhere on GPU, by more than its 33%
  flop saving predicts -- the scaled branch also syncs log|det| to the host
  every Newton step. Halley has no scaled variant at all, which is a second
  reason to expect it to do relatively better here.""")


## What the runs so far measured

**SDC beat the real incumbent at n=256 and n=512, and lost at n=1024.** Against
`cupy.linalg.eig`, best configuration at each size, `scale=none` (OPTIMIZATION_LOG #38,
after the device-leaf work):

| n | `cupy.linalg.eig` | SDC | | leaf |
|---|---|---|---|---|
| 256 | 87.3 ms | 63.0 ms | **1.39×** | host |
| 512 | 259.2 ms | 224.7 ms | **1.15×** | auto |
| 1024 | 786.3 ms | 997.8 ms | 0.79× | auto |

Device leaves are a real gain where they apply — n=1024 went from 1271.5 ms
(#36, host leaves) to 997.8, a **1.27× improvement** — but not enough to
overturn the size trend.

That is the campaign's **first outright win over a vendor eigensolver on any
substrate** — narrow, size-limited, and on the nonsymmetric side exactly where
OPTIMIZATION_LOG #24 predicted the opening would be.

**But both winning sizes are powers of two, and that is now a known trap.**
OPTIMIZATION_LOG #41 measured an **88% cache-aliasing penalty in `dgeev` at exactly
n=512** and 19% at n=1024. On CPU, off the powers of two, the best SDC variant
*lost* 1.35× at n=500 and 1.54× at n=1000 — while *appearing* to win 1.31× at
n=512, entirely on the incumbent's penalty. Whether cuSOLVER's `xgeev` behaves
the same way is **not established and is not claimed**. Cell 3a now measures it
in eight lines, and cell 3b brackets every size with an odd neighbour and
prints the inflation factor. If 3a is flat the wins above stand as measured; if
it spikes, the odd-*n* rows are the only honest ones.

**Accuracy degrades with n and needs watching**: 3.8e-14 → 2.9e-13 → 3.4e-11
against `cupy.linalg.eig`'s 5.2e-15 → 9.2e-15 → 1.3e-14. The n=1024 end of
that was the visible symptom of a real defect — the split gate was **five
orders of magnitude too loose** (#39, below) — and tightening it improved the
n=1024 split 99×. The remaining trend is still the wrong way and still
unexplained.

**Watch for a fallback.** The `splits` column and any `NOT ACCURATE` row tell
you whether SDC did its work or quietly fell back to `eigvals`. A fallback that
returns the right answer fast is measuring LAPACK, not SDC.

## The far-field map is a hardware choice

New in this revision, and the reason cell 1 matters. The sign iteration's
far-field step can be Newton (one inverse, quadratic) or Halley (one inverse
plus three gemms, cubic). Halley needs a fraction ρ of Newton's steps, so it
wins exactly when

```
inv/gemm  >  3ρ/(1 − ρ)
```

Measured step counts on CPU, Newton → Halley:

| class | n=200 | n=400 | ρ | threshold |
|---|---|---|---|---|
| Ginibre | 15 → 10 | 13 → 8 | 0.62 | **4.9** |
| near-symmetric | 7 → 5 | 13 → 8 | 0.55 | 3.7 |
| symmetric | 8 → 4 | 9 → 5 | 0.53 | **3.4** |

This CPU measures `inv/gemm = 3.90`, which lands *between* the Ginibre and
near-symmetric thresholds — and that is precisely the split #41 measured end to
end: **0.92×–1.01× on Ginibre, 1.11×–1.13× on near-symmetric.** The model
predicts both signs correctly from one hardware ratio, which is why it is worth
extrapolating to a different substrate rather than re-deriving.

A GPU should raise `inv/gemm` substantially: gemm runs at or near peak while
`getrf`/`getri` are panel-bound and latency-exposed. If it does, Halley stops
being noise. Cell 1 prints the ratio and the verdict, cell 3 races it on both
matrix classes, and `ns_order ∈ {3,5,7}` exposes the gemm-only endgame for the
same reason. **Halley also never syncs `log|det|` to the host**, which is a
second and independent reason to expect it to do relatively better here — the
same sync is why `scale=none` beat `scale=det` by more than its 33% flop saving
predicted.

## What this notebook does not settle

* **fp32.** The whole sign iteration could run in single precision with a final
  refinement, and on a card where fp32 is 14.6× fp64 that is the largest
  untouched lever. It is untested here because the refinement ladder's behaviour
  on *non-symmetric* input is itself unmeasured — OPTIMIZATION_LOG #24 found the ladder
  loses its Newton–Schulz half there (non-symmetric eigenvectors are not
  orthonormal, so re-orthonormalising would destroy the answer), and only the
  consult-A half survives.
* **Batched small blocks.** `leaf=deep` recurses to 2×2 and issues one kernel
  per block at the bottom levels — 526 splits at n=1024. Level *k* holds 2^k
  blocks of the same size, so they could be batched, which is what made SSJ-BC
  viable on GPU in the companion notebook. Untested, and it is the obvious fix
  if `deep` loses on launch overhead rather than on arithmetic.
* **The inverse-free variant (Bai–Demmel–Gu).** Deliberately *not* offered. It
  was refuted analytically and unconditionally: one IRS step is 13.33n³ against
  Newton's 2n³, so 15 steps is 200n³, which at *perfect* gemm efficiency already
  exceeds `dgeev`'s entire 93.9 gemm-equivalents. That refutation is in flops, so
  it survives the change of substrate. Its *stability* claim did check out
  though — it converges on cond(V)=1e6 where the shipped iteration stalls and
  never converges in 80 iterations — so it is a fallback, not a speed lever.
* **Eigenvectors.** This computes eigenvalues only, as `ssj.sdc` does. The
  orthogonal similarities are accumulated implicitly and thrown away; keeping
  them costs one more gemm per split.

## Provenance

Ported from `ssj.sdc` and validated on the NumPy path against `numpy.linalg.eig`
across Ginibre, near-symmetric and companion matrices at n = 128, 250 and 256,
over **all 108 combinations** of `far ∈ {newton, halley}`, `ns_order ∈ {3,5,7}`
and `leaf_solver ∈ {auto, deep}` — every one inside 1e-8, worst case 1.9e-12,
with `fallbacks == 0` asserted so no configuration is silently measuring LAPACK.

Findings built in and commented at the point of use:

* **#31** the Frobenius handoff bound — a fixed threshold in the wrong norm made
  symmetric input 54× slower than `dgeev` (2712 ms against 48 ms at n=400).
  Since ‖M‖₂ ≤ ‖M‖_F, gating on ‖I−X²‖_F < 1 is guaranteed safe, and in
  normalized units that is a 1/√n **scaling law, not a constant**.
* **#30** the far-field gate — Δ = X⁻¹(I−X²)/2, so the update norm is the same
  convergence signal for O(n²), and the 2n³ test gemm is formed once or twice
  instead of 8–11 times.
* **#28** the 3n/5 leaf — n/2 is too *small*, because the centred split returns
  r near but never on n/2, so one half comes back a few rows too big and buys a
  whole second full-size sign iteration.
* **#39** the split gate is **1e-11, not 1e-6**. The old value never fired:
  headroom against what the split actually achieves ran 25,892× at n=1024 and
  17 million× at n=256, so a genuinely bad split sailed through. The retry loop
  was already the right machinery; only the threshold was wrong.
* **#41** pivoted QR runs at **6.0% of gemm rate** (11.04 gemm-equivalents),
  making it the worst-performing primitive in the whole method — worse per flop
  than `dgeev` itself. CuPy has no pivoted QR, so this port was forced onto the
  randomized range-finder (9.45), which turns out to be the better kernel. The
  CPU version still uses `dgeqp3`; there the saving is only 1.3% of the total.
